# 51. 감사추적 반려 케이스 기록 보완

## 목적
지금 debate_rounds는 토의가 "승인"된 경우만 history에 저장되고,
"반려"나 "에스컬레이트"된 경우엔 정작 가장 궁금한 근거(critic이 왜
막았는지)가 기록에서 빠지는 구조적 공백이 있다. 이걸 고쳐서 성공/실패
관계없이 모든 토의 왕복이 감사추적 리포트에 남도록 만든다.

## 배경
- 노트북 45-50에서 Aliphatic_long_chain 완전해결, 선례 라이브러리 24건,
  LLM 병렬화, 토의 기반 검증 에이전트(debate), 도킹 자동화(4개 표적),
  scoring/활성보존 지표 연동, 다중문제 충돌조정, 감사추적 리포트까지
  순차적으로 구축 완료
- 노트북 50 말미: quota 리셋 직후 200개 실API 통합 테스트에서
  Michael_acceptor_1/EGFR 도킹이 규칙과 무관한데도 critic이 "이 분자는
  EGFR 표적"이라고 잘못 단정하는 시스템적 편향 발견, DOCKING_TARGETS에
  caveat 필드 추가해서 수정·검증 완료
- 그 과정에서 반려 케이스 5건을 직접 재현해서 확인했는데, 정작 배치
  실행 시 저장되는 history엔 그 근거가 하나도 안 남는 걸 확인 —
  이번 세션의 핵심 동기

## 이번 세션 목표
1. molecule_editor.py: 토의 결과(승인/반려/에스컬레이트) 관계없이
   모든 시도를 스텝 단위로 기록하도록 수정
2. audit.py: 반려/에스컬레이트된 토의 왕복도 리포트에 표시하도록 확장
3. 회귀 테스트 + 어제 나온 반려 5건 케이스로 실제 리포트 확인

## 추가로 고려할 만한 것 (사용자가 확장 중: 다단계 사용 지원 + 질환별 세분화)
- 반복적으로 발생한 버그(함수 중복 정의, 잘못된 branch 원자 선택 등)를
  미연에 잡을 수 있는 자동 회귀 테스트 스위트(pytest) 도입
- 승인/반려/에스컬레이트에 신뢰도(confidence) 점수를 붙여서, 사람 검토
  큐를 "가장 애매한 것부터" 우선순위화
- 리포트마다 그 판단에 쓰인 선례 라이브러리 버전/모델 버전을 명시하는
  재현성 매니페스트(오픈소스 공개·논문화 시 중요)
- 오픈소스 공개 전 라이선스/출처 정리(RDKit, sascorer, ChEMBL/GtoPdb
  데이터, AutoDock Vina 등 외부 의존성 고지)

## 작업 스타일 (선호)
- 수정 사항이 20줄 미만이고 새 항목 추가가 아니면 전체 코드 블록 대신 삽입 위치만 설명
- 재로드 + 검증 코드는 항상 하나의 셀로 통합해서 제공
- 파일 수정 후에는: 문법 검증 → 재로드 → 확인 → 커밋

In [1]:
# 셀1 - install
!pip install rdkit -q
!pip install chembl_webresource_client -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q
!apt-get install -y openbabel -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 118337 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpackin

In [2]:
# 셀2 - github token + clone + cd + pwd
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 828, done.
remote: Counting objects: 100% (288/288), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 828 (delta 180), reused 206 (delta 104), pack-reused 540 (from 1)
Receiving objects: 100% (828/828), 7.19 MiB | 11.65 MiB/s, done.
Resolving deltas: 100% (497/497), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀3 - git config
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀4 - import + 데이터 로드 + vina 설치 (ChEMBL 제외 — EBI 서버 장애로 오늘은 건너뜀)
import importlib, json, ast, time, os, random
from collections import Counter
from rdkit import Chem
import requests

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.precedent_library
import src.tools.agent
import src.tools.docking
import src.tools.activity_metrics
import src.tools.audit
import models.tox_baseline

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, batch_iterative_fix_loop, clear_failure_memory
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents
from src.tools.docking import auto_dock_precedent, inspect_hetatm, DOCKING_TARGETS
from src.tools.activity_metrics import compute_activity_preservation_metrics, classify_activity_risk_v3
from src.tools.audit import generate_audit_report
from models.tox_baseline import train_tox21_baseline, make_tox_predictor

data = load_tox21_clean(random_state=7)

base_url = "https://www.guidetopharmacology.org/services"
def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()
def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina_bin
!chmod +x vina_bin

import urllib.request, sys
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py", "sascorer.py")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz", "fpscores.pkl.gz")
sys.path.append('.')
import sascorer

from src.tools.agent import set_sascorer_module, set_tox_predictor
set_sascorer_module(sascorer)
tox_model = train_tox21_baseline(data)
set_tox_predictor(make_tox_predictor(tox_model))

print(f"선례 수: {len(PRECEDENT_LIBRARY)} (24여야 정상)")
print("vina_bin 존재:", os.path.exists('vina_bin'))
print("caveat 필드 반영 여부:", 'caveat' in open('src/tools/docking.py').read())
print("debate_rounds 반영 여부:", 'debate_rounds' in open('src/tools/molecule_editor.py').read())

[13:06:19] WARNING: not removing hydrogen atom without neighbors
[13:06:19] Explicit valence for atom # 8 Al, 6, is greater than permitted
[13:06:19] Explicit valence for atom # 3 Al, 6, is greater than permitted
[13:06:19] Explicit valence for atom # 4 Al, 6, is greater than permitted
[13:06:20] Explicit valence for atom # 4 Al, 6, is greater than permitted
[13:06:20] Explicit valence for atom # 9 Al, 6, is greater than permitted
[13:06:20] Explicit valence for atom # 5 Al, 6, is greater than permitted
[13:06:21] Explicit valence for atom # 16 Al, 6, is greater than permitted
[13:06:21] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[13:06:23] WARNING: not removing hydrogen atom without neighbors


선례 수: 24 (24여야 정상)
vina_bin 존재: True
caveat 필드 반영 여부: True
debate_rounds 반영 여부: True


In [5]:
# 셀5 - Qwen 연결 확인
from openai import OpenAI

dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(
    api_key=dashscope_key,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
    timeout=30,
)
QWEN_MODEL = "qwen3.8-max"

try:
    response = client_qwen.chat.completions.create(
        model=QWEN_MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=10,
    )
    print("Qwen 연결 확인:", response.choices[0].message.content)
except Exception as e:
    print("Qwen 연결 실패:", repr(e))

Qwen 연결 확인: Hi! How can I help you today?


In [10]:
import os
os.makedirs('tests', exist_ok=True)
print("tests/ 폴더 생성 완료:", os.path.exists('tests'))

tests/ 폴더 생성 완료: True


In [11]:
%%writefile tests/test_regression.py
"""회귀 방지 테스트 스위트. 반복적으로 겪은 문제들
(함수 중복 정의, 수정이 실제로 반영 안 됨, 핵심 파이프라인 깨짐)을
매 세션 시작 시 한 번에 잡아내기 위한 것. 네트워크/API 호출 없는
테스트만 포함(빠르게, 매번 돌릴 수 있게)."""

import ast
import glob
import os
import pytest

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def _all_source_files():
    patterns = ["src/tools/*.py", "src/models/*.py", "models/*.py"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(REPO_ROOT, pat)))
    return files


def test_no_duplicate_top_level_function_definitions():
    """%%writefile -a로 같은 함수를 두 번 추가해서, 나중(옛날) 버전이
    최종 반영되는 사고(batch_iterative_fix_loop 사례)를 방지."""
    problems = []
    for path in _all_source_files():
        with open(path) as f:
            source = f.read()
        tree = ast.parse(source)
        names = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
        seen = set()
        for name in names:
            if name in seen:
                problems.append(f"{path}: '{name}' 중복 정의")
            seen.add(name)
    assert not problems, "\n".join(problems)


def test_all_source_files_parse():
    """모든 소스 파일이 문법적으로 유효한지."""
    problems = []
    for path in _all_source_files():
        with open(path) as f:
            try:
                ast.parse(f.read())
            except SyntaxError as e:
                problems.append(f"{path}: {e}")
    assert not problems, "\n".join(problems)


def test_precedent_library_structure():
    from src.tools.precedent_library import PRECEDENT_LIBRARY
    assert len(PRECEDENT_LIBRARY) >= 24, f"선례 수가 예상보다 적음: {len(PRECEDENT_LIBRARY)}"
    required_keys = {"rule", "type", "description"}
    for i, p in enumerate(PRECEDENT_LIBRARY):
        missing = required_keys - p.keys()
        assert not missing, f"{i}번째 항목에 키 누락: {missing}"


def test_docking_targets_have_caveat_field():
    """caveat 필드 누락 회귀 방지 (오늘 겪은 EGFR 편향 사고)."""
    from src.tools.docking import DOCKING_TARGETS
    for rule, info in DOCKING_TARGETS.items():
        assert "caveat" in info, f"{rule}에 caveat 필드 없음"


def test_agent_required_functions_exist():
    """agent.py에 있어야 할 핵심 함수/설정 함수들이 다 있는지."""
    import src.tools.agent as agent
    required = [
        "ask_llm_which_problem_to_fix", "ask_llm_which_candidate_to_use",
        "ask_llm_debate_fix", "should_debate",
        "set_debate_budget", "set_sascorer_module", "set_tox_predictor",
        "_try_get_docking_evidence", "_try_compute_score", "_try_get_activity_risk",
    ]
    missing = [name for name in required if not hasattr(agent, name)]
    assert not missing, f"agent.py에 없는 함수: {missing}"


def test_batch_iterative_fix_loop_signature_has_debate_params():
    import inspect
    from src.tools.molecule_editor import batch_iterative_fix_loop
    sig = inspect.signature(batch_iterative_fix_loop)
    assert "use_debate" in sig.parameters
    assert "debate_max_rounds" in sig.parameters


def test_molecule_editor_debate_rounds_recorded_in_history():
    """감사추적용 debate_rounds 키가 iterative_fix_loop 코드에 존재하는지
    (실제 토의 왕복 기록 여부는 별도 통합테스트에서 확인)."""
    with open(os.path.join(REPO_ROOT, "src/tools/molecule_editor.py")) as f:
        content = f.read()
    assert "debate_rounds" in content


@pytest.mark.slow
def test_smoke_iterative_fix_loop_resolves_simple_chain():
    """규칙 기반(LLM 없음) 스모크 테스트: 가장 기본적인 회귀 방지."""
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    clear_failure_memory()
    r = iterative_fix_loop("CCCCCCCCCCCCCCCC", max_iterations=10, candidate_idx=0)
    assert r["status"] == "success"


@pytest.mark.slow
def test_smoke_conflict_resolution_picks_lower_remaining_count():
    """다중문제 충돌조정이 실제로 남는 문제 수 적은 쪽을 먼저 고르는지."""
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    clear_failure_memory()
    smi = "NNC(=O)CP(=O)(c1ccccc1)c1ccccc1"  # hydrazine + phosphor
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    step1 = r["history"][1]
    assert step1.get("fixed_rule") == "hydrazine"
    assert "충돌 조정" in step1.get("problem_reason", "")

Writing tests/test_regression.py


In [12]:
!pip install pytest -q
!cd /content/laidd-2026 && python -m pytest tests/test_regression.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/laidd-2026
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 9 items                                                              

tests/test_regression.py::test_no_duplicate_top_level_function_definitions PASSED [ 11%]
tests/test_regression.py::test_all_source_files_parse PASSED             [ 22%]
tests/test_regression.py::test_precedent_library_structure PASSED        [ 33%]
tests/test_regression.py::test_docking_targets_have_caveat_field PASSED  [ 44%]
tests/test_regression.py::test_agent_required_functions_exist PASSED     [ 55%]
tests/test_regression.py::test_batch_iterative_fix_loop_signature_has_debate_params PASSED [ 66%]
tests/test_regression.py::test_molecule_editor_debate_rounds_recorded_in_history PASSED [ 77%]
tests/test_regression.py::test_smoke_iterativ

In [13]:
import ast
with open('tests/test_regression.py') as f:
    ast.parse(f.read())
print("✅ test_regression.py 문법 정상")

✅ test_regression.py 문법 정상


In [ ]:
!cat src/tools/audit.py

In [16]:
%%writefile src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib
from src.tools.agent import (ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use,
                                  ask_llm_debate_fix, should_debate)

def _library_version_hash():
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]
            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue
            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue
            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')
            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def _candidate_order_for_rule(rule_name: str, preferred_idx: int):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return [preferred_idx]
    n = len(info['candidates'])
    order = [preferred_idx] if 0 <= preferred_idx < n else []
    order += [i for i in range(n) if i != preferred_idx]
    return order


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini",
                        use_failure_memory: bool = True, use_debate: bool = False,
                        debate_max_rounds: int = 2):
    """진단->치환->재평가를 반복.
    핵심: candidate가 '화학적으로 유효(is_valid)'해도 대상 규칙이 실제로
    해소됐는지 재진단(detect_toxicophores)까지 확인한다. 그렇지 않으면
    항상 valid하지만 문제를 안 고치는 candidate(예: 단순 삽입형)가
    무한 반복 채택되어 진짜 해법(예: 분기형)으로 넘어가지 못하는 문제가
    있었음. 완전 해소가 안 되면 마지막으로 시도한(=대개 더 나은)
    valid 결과를 fallback으로 채택해 다음 iteration에서 계속 개선."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            # 다중 문제 충돌 조정: 각 문제를 먼저 고쳤을 때 남는 전체
            # toxicophore 수가 가장 적어지는 순서로 정렬(그리디, LLM 미사용).
            sim_scores = {}
            for p in known_problems:
                rn = p['rule_name']
                try:
                    trial = propose_fix(current, rn, candidate_idx)
                    if trial is None or not trial.get('is_valid'):
                        sim_scores[rn] = 999
                        continue
                    remaining = detect_toxicophores(trial['new_smiles'])
                    sim_scores[rn] = len(remaining)
                except Exception:
                    sim_scores[rn] = 999
            ordered_rules = sorted(sim_scores, key=sim_scores.get)
            problem_reason = f"규칙 기반(충돌 조정: 남는 문제 수 적은 순 - {sim_scores})"

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            all_debate_logs_for_step = []
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                preferred_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if preferred_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                preferred_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도)"

            rule_fixed = None
            fallback_attempt = None
            fallback_used_idx = None
            fallback_reason = None

            for try_idx in _candidate_order_for_rule(candidate_rule, preferred_candidate_idx):
                memory_key = (current, candidate_rule, try_idx, _library_version_hash())
                if use_failure_memory and memory_key in _FAILURE_MEMORY:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](memory-skip)")
                    continue

                attempt = propose_fix(current, candidate_rule, try_idx)
                if attempt is None or not attempt.get('is_valid'):
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}]")
                    if use_failure_memory:
                        _FAILURE_MEMORY[memory_key] = True
                    continue

                # 파괴적 편집 가드: 무거운 원자 50% 이상이 사라지면 "고침"이
                # 아니라 분자 자체를 파괴한 것으로 간주(remove_substituent류
                # 편집이 비고리 분자의 대부분을 통째로 잘라내는 사고 방지)
                mol_current_check = Chem.MolFromSmiles(current)
                mol_new_check = Chem.MolFromSmiles(attempt['new_smiles'])
                if mol_current_check and mol_new_check:
                    atoms_before = mol_current_check.GetNumHeavyAtoms()
                    atoms_after = mol_new_check.GetNumHeavyAtoms()
                    loss_ratio = 1 - (atoms_after / atoms_before) if atoms_before > 0 else 0
                    if loss_ratio >= 0.5:
                        failed_attempts.append(
                            f"{candidate_rule}[idx={try_idx}](파괴적 편집 거부: 원자 {loss_ratio:.0%} 손실)"
                        )
                        if use_failure_memory:
                            _FAILURE_MEMORY[memory_key] = True
                        continue

                # valid해도 실제로 이 규칙이 재진단에서 사라졌는지 확인
                recheck = detect_toxicophores(attempt['new_smiles'])
                still_flagged = any(p['rule_name'] == candidate_rule for p in recheck)

                if not still_flagged:
                    candidate_obj = get_replacement_candidates(candidate_rule)['candidates'][try_idx]
                    debate_suffix = ""

                    if use_debate and llm_client is not None and should_debate(candidate_obj.get('rationale', '')):
                        debate_result = ask_llm_debate_fix(
                            llm_client, llm_model, current, attempt['new_smiles'], candidate_rule,
                            candidate_obj['name'], candidate_obj.get('rationale', ''),
                            client_type=llm_client_type, max_rounds=debate_max_rounds,
                        )
                        if debate_result['final_verdict'] == 'rejected':
                            all_debate_logs_for_step.append({
                                "rule": candidate_rule, "candidate_idx": try_idx,
                                "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                            })
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 결과 반려)")
                            if use_failure_memory:
                                _FAILURE_MEMORY[memory_key] = True
                            continue
                        elif debate_result['final_verdict'] == 'escalate':
                            all_debate_logs_for_step.append({
                                "rule": candidate_rule, "candidate_idx": try_idx,
                                "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                            })
                            flagged_for_review.add(candidate_rule)
                            if candidate_rule not in skipped_rules:
                                skipped_rules.append(candidate_rule)
                            skipped_details.append({
                                "rule_name": candidate_rule,
                                "reason": f"LLM 토의가 {debate_max_rounds}라운드 안에 합의에 도달하지 못해 "
                                          f"사람 검토로 넘김 (마지막 논쟁: {debate_result['rounds'][-1]['text']})",
                                "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                            })
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 합의 실패, escalate)")
                            all_debate_logs_for_step.append({
                                "rule": candidate_rule, "candidate_idx": try_idx,
                                "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                            })
                            continue
                        all_debate_logs_for_step.append({
                            "rule": candidate_rule, "candidate_idx": try_idx,
                            "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                        })
                        debate_suffix = " (토의 승인)"

                    rule_fixed = attempt
                    candidate_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 완전 해소){debate_suffix}"
                    break
                else:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](valid이나 미해소)")
                    fallback_attempt = attempt
                    fallback_used_idx = try_idx
                    fallback_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 부분 개선/다음 iteration에서 계속)"

            if rule_fixed is None and fallback_attempt is not None:
                rule_fixed = fallback_attempt
                candidate_reason = fallback_reason

            if rule_fixed is not None:
                fixed = rule_fixed
                target_rule = candidate_rule
                break

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 "
                              f"({failed_attempts}) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 "
                              f"실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 "
                              f"등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙/candidate {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
            "debate_rounds": all_debate_logs_for_step,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}


def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True, use_debate=False,
                               debate_max_rounds=2):
    """여러 분자에 iterative_fix_loop를 스레드 병렬로 적용.
    LLM API 호출이 병목인 경우(네트워크 대기 시간) 유효한 개선이며,
    화학 계산 로직(iterative_fix_loop 자체)은 전혀 수정하지 않는다.
    반환: [(smiles, result_dict), ...] (완료 순서, 입력 순서와 다를 수 있음)
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def _process_one(smi):
        r = iterative_fix_loop(
            smi, max_iterations=max_iterations, candidate_idx=candidate_idx,
            llm_client=llm_client, llm_model=llm_model, llm_client_type=llm_client_type,
            use_debate=use_debate, debate_max_rounds=debate_max_rounds,
        )
        return smi, r

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_process_one, smi): smi for smi in smiles_list}
        for i, future in enumerate(as_completed(futures)):
            smi, r = future.result()
            results.append((smi, r))
            if progress:
                print(f"[{i+1}/{len(smiles_list)}] {smi[:30]} -> {r['status']}")
    return results




Overwriting src/tools/molecule_editor.py


In [16]:
%%writefile src/tools/audit.py
"""iterative_fix_loop 결과를 사람이 읽기 좋은 감사추적 리포트로 변환.
심사/발표 자료용 — AI가 왜 그렇게 판단했는지, 언제 사람 검토로 넘겼는지를
그대로 보여준다."""


def generate_audit_report(result, original_smiles=None):
    lines = []
    lines.append("=" * 60)
    lines.append("치환 감사추적 리포트")
    lines.append("=" * 60)
    if original_smiles:
        lines.append(f"원본 분자: {original_smiles}")
    lines.append(f"최종 상태: {result['status']}")
    lines.append(f"최종 분자: {result.get('final_smiles', '')}")
    lines.append("")

    lines.append("--- 단계별 이력 ---")
    for h in result.get('history', []):
        step = h.get('step')
        lines.append(f"\n[스텝 {step}] {h.get('smiles', '')}")
        problems = h.get('problems')
        if problems:
            rule_names = [p['rule_name'] for p in problems]
            lines.append(f"  진단된 문제: {rule_names}")
        if 'fixed_rule' in h:
            lines.append(f"  고친 규칙: {h['fixed_rule']} (판단 근거: {h.get('problem_reason', '')})")
            lines.append(f"  적용된 치환: {h.get('candidate_used', '')}")
            lines.append(f"  candidate 선택 근거: {h.get('candidate_reason', '')}")
            debate_attempts = h.get('debate_rounds')
            if debate_attempts:
                lines.append("  --- 토의(debate) 시도 기록 ---")
                for attempt in debate_attempts:
                    lines.append(f"    ▸ {attempt['rule']}[idx={attempt['candidate_idx']}] 최종: {attempt['verdict']}")
                    for r in attempt['rounds']:
                        role = r.get('role')
                        round_num = r.get('round')
                        text = r.get('text', {})
                        if role == 'critic':
                            lines.append(f"      [R{round_num} critic] {text.get('verdict')}: {text.get('reason', '')}")
                        else:
                            lines.append(f"      [R{round_num} proposer] {text.get('stance')}: {text.get('argument', '')}")

    if result.get('skipped_rules'):
        lines.append("\n--- 처리 못 하고 넘긴 규칙 (라이브러리 미등록 또는 사람 검토 필요) ---")
        for detail in result.get('skipped_details', []):
            lines.append(f"  - {detail['rule_name']}: {detail['reason']}")

    if result['status'] == 'stuck':
        lines.append(f"\n--- stuck 사유 ---\n{result.get('reason_detail', result.get('reason', ''))}")

    lines.append("=" * 60)
    return "\n".join(lines)


Overwriting src/tools/audit.py


In [22]:
content = open('src/tools/molecule_editor.py').read()
lines = content.split('\n')

for i, line in enumerate(lines):
    if "for candidate_rule in ordered_rules:" in line:
        for j in range(i, min(len(lines), i+3)):
            marker = ">>> " if j == i else "    "
            print(f"{marker}{j+1}: {lines[j]}")

>>> 223:         for candidate_rule in ordered_rules:
    224:             all_debate_logs_for_step = []
    225:             if llm_client is not None:


In [23]:
import ast
with open('src/tools/molecule_editor.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ molecule_editor.py 문법 정상")
print("✅ 옛날 변수명(debate_log_for_step) 완전히 사라졌는지:", 'debate_log_for_step' not in content, "(True여야 정상)")
print("✅ all_debate_logs_for_step 개수:", content.count('all_debate_logs_for_step'), "(5곳이어야 정상: 초기화1 + rejected1 + escalate1 + approved1 + history기록1)")

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
importlib.reload(src.tools.agent)
importlib.reload(src.tools.audit)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
from src.tools.audit import generate_audit_report
clear_failure_memory()

r = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert r['status'] == 'success'
print("✅ 회귀 없음 확인")

r2 = iterative_fix_loop(
    'C=C(CC(=O)O)C(=O)O', max_iterations=10, candidate_idx=0,
    llm_client=client_qwen, llm_model=QWEN_MODEL, llm_client_type="openai_compatible",
    use_debate=True, debate_max_rounds=2,
)
print("\n" + generate_audit_report(r2, original_smiles='C=C(CC(=O)O)C(=O)O'))

✅ molecule_editor.py 문법 정상
✅ 옛날 변수명(debate_log_for_step) 완전히 사라졌는지: True (True여야 정상)
✅ all_debate_logs_for_step 개수: 6 (5곳이어야 정상: 초기화1 + rejected1 + escalate1 + approved1 + history기록1)
✅ 회귀 없음 확인

치환 감사추적 리포트
원본 분자: C=C(CC(=O)O)C(=O)O
최종 상태: success
최종 분자: CC(CC(=O)O)C(=O)O

--- 단계별 이력 ---

[스텝 0] C=C(CC(=O)O)C(=O)O
  진단된 문제: ['Michael_acceptor_1']

[스텝 1] CC(CC(=O)O)C(=O)O
  고친 규칙: Michael_acceptor_1 (판단 근거: 유일한 치환 가능 후보)
  적용된 치환: saturated (C-C single bond)
  candidate 선택 근거: 제시된 분자는 에타크린산과 같은 승인된 공유결합 억제제 골격이 아닌 단순 알파,베타-불포화 다이카르복실산으로 비특이적 Michael addition 독성 위험이 있으므로 참고사항의 예외에 해당하지 않아 이중결합 환원이 필요함. (candidate_idx=0, 완전 해소) (토의 승인)
  --- 토의(debate) 시도 기록 ---
    ▸ Michael_acceptor_1[idx=0] 최종: approved
      [R1 critic] approved: 제안된 치환은 알파,베타-불포화 카르보닐(Michael acceptor)을 포화 결합으로 환원하여 비특이적 공유결합 및 세포 독성 위험을 효과적으로 제거하였으며, 이는 신약개발 초기 단계에서 안전성 프로파일 개선을 위한 표준적이고 타당한 전략입니다. 제안자가 언급한 에타크린산 등의 예외는 해당 분자가 EGFR 등 특정 공유결합 표적을 겨냥한다는 명확한 증거가 있을 때만 유효하므로, 현재와 같이 표적 특이성이 입증되지 않은 상태에서는 PAINS 경고 해소가 

In [24]:
!cd /content/laidd-2026 && python -m pytest tests/test_regression.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/laidd-2026
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 9 items                                                              

tests/test_regression.py::test_no_duplicate_top_level_function_definitions PASSED [ 11%]
tests/test_regression.py::test_all_source_files_parse PASSED             [ 22%]
tests/test_regression.py::test_precedent_library_structure PASSED        [ 33%]
tests/test_regression.py::test_docking_targets_have_caveat_field PASSED  [ 44%]
tests/test_regression.py::test_agent_required_functions_exist PASSED     [ 55%]
tests/test_regression.py::test_batch_iterative_fix_loop_signature_has_debate_params PASSED [ 66%]
tests/test_regression.py::test_molecule_editor_debate_rounds_recorded_in_history PASSED [ 77%]
tests/test_regression.py::test_smoke_iterativ

In [25]:
!cd /content/laidd-2026 && git add -A && git commit -m "Fix audit trail gap: debate rounds now recorded for all attempts (approved/rejected/escalated), not just the final approved one" && git push

[main 145835b] Fix audit trail gap: debate rounds now recorded for all attempts (approved/rejected/escalated), not just the final approved one
 3 files changed, 146 insertions(+), 15 deletions(-)
 create mode 100644 tests/test_regression.py
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (8/8), 3.02 KiB | 3.02 MiB/s, done.
Total 8 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   6d239e6..145835b  main -> main


In [6]:
!cd /content/laidd-2026 && python -m pytest tests/test_regression.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/laidd-2026
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 9 items                                                              

tests/test_regression.py::test_no_duplicate_top_level_function_definitions PASSED [ 11%]
tests/test_regression.py::test_all_source_files_parse PASSED             [ 22%]
tests/test_regression.py::test_precedent_library_structure PASSED        [ 33%]
tests/test_regression.py::test_docking_targets_have_caveat_field PASSED  [ 44%]
tests/test_regression.py::test_agent_required_functions_exist PASSED     [ 55%]
tests/test_regression.py::test_batch_iterative_fix_loop_signature_has_debate_params PASSED [ 66%]
tests/test_regression.py::test_molecule_editor_debate_rounds_recorded_in_history PASSED [ 77%]
tests/test_regression.py::test_smoke_iterativ

In [7]:
!cd /content/laidd-2026 && git add -A && git commit -m "Fix audit trail gap: debate rounds now recorded for all attempts (approved/rejected/escalated), not just the final approved one" && git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [8]:
!cd /content/laidd-2026 && git log --oneline -15

145835b (HEAD -> main, origin/main, origin/HEAD) Fix audit trail gap: debate rounds now recorded for all attempts (approved/rejected/escalated), not just the final approved one
6d239e6 Add caveat field to docking targets lacking rule-specific relevance (Michael_acceptor_1/EGFR); critic no longer falsely asserts specific target mechanism
9c89b1d Pass use_debate/debate_max_rounds through batch_iterative_fix_loop
09f35b1 Document phosphor rule limitation: triarylphosphine oxides have no approved-drug precedent, defer new candidate (null-result)
c141d87 Connect activity preservation risk assessment into debate critic prompt via _try_get_activity_risk
4c20dce Consolidate activity preservation metrics (compute_activity_preservation_metrics, classify_activity_risk_v3) into src/tools/activity_metrics.py, reusing fingerprint from models/tox_baseline.py
a146f45 Add greedy multi-problem conflict resolution (fewest-remaining-problems ordering) and audit trail report generator
591076d Add Tox21 bas

In [9]:
!cd /content/laidd-2026 && git show --stat 33cb88c
!cd /content/laidd-2026 && git show --stat 96fafa2

commit 33cb88c381e74441ca3f52d8e96594f233b5e036
Author: Dec32th <hyekyeong.w@gmail.com>
Date:   Thu Aug 13 13:25:54 2026 +0000

    Add docking.py: automated docking pipeline with cached results and mandatory human-confirmed HETATM handling

 outputs/docking_cache.json        | 10 +++++++++
 src/tools/docking.py              |  4 ++--
 targets/catechol_fixed_log.txt    | 47 +++++++++++++++++++++++++++++++++++++++
 targets/catechol_original_log.txt | 47 +++++++++++++++++++++++++++++++++++++++
 4 files changed, 106 insertions(+), 2 deletions(-)
commit 96fafa237aaa7e0445e1b5adc23cd16b5e93b45e
Author: Dec32th <hyekyeong.w@gmail.com>
Date:   Thu Aug 13 13:19:59 2026 +0000

    Add docking.py: automated docking pipeline with cached results and mandatory human-confirmed HETATM handling

 src/tools/docking.py | 177 +++++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 177 insertions(+)


In [6]:
import random, time
from collections import Counter

random.seed(7)
sample_200 = random.sample(list(data['smiles_valid']), 200)

from src.tools.agent import _llm_error_log
_llm_error_log.clear()
clear_failure_memory()

t0 = time.time()
results_debate_v2 = batch_iterative_fix_loop(
    sample_200, max_iterations=10, candidate_idx=0,
    llm_client=client_qwen, llm_model=QWEN_MODEL, llm_client_type="openai_compatible",
    max_workers=8, progress=False,
    use_debate=True, debate_max_rounds=2,
)
elapsed = time.time() - t0

status_counter_v2 = Counter(r['status'] for _, r in results_debate_v2)
print(f"200개, use_debate=True(caveat 수정 후), max_workers=8: {elapsed:.1f}초")
print(f"에러: {len(_llm_error_log)}건")
for status, count in status_counter_v2.most_common():
    print(f"{status}: {count}")

# 규칙 기반(use_debate=False)과 비교
clear_failure_memory()
results_rule_v2 = batch_iterative_fix_loop(
    sample_200, max_iterations=10, candidate_idx=0,
    max_workers=8, progress=False,
    use_debate=False,
)
status_counter_rule = Counter(r['status'] for _, r in results_rule_v2)
print("\n--- 규칙 기반 ---")
for status, count in status_counter_rule.most_common():
    print(f"{status}: {count}")

# 차이 분석
results_debate_dict = dict(results_debate_v2)
results_rule_dict = dict(results_rule_v2)
diffs = [(smi, results_rule_dict[smi]['status'], results_debate_dict[smi]['status'])
         for smi in sample_200 if results_rule_dict[smi]['status'] != results_debate_dict[smi]['status']]
print(f"\n총 {len(diffs)}/200건 차이")
for smi, rb, d in diffs:
    print(f"[{rb} -> {d}] {smi}")

# Michael_acceptor_1 토의 승인/반려 분포 확인 (caveat 효과 검증 핵심)
ma1_verdicts = Counter()
for _, r in results_debate_v2.items() if isinstance(results_debate_v2, dict) else results_debate_v2:
    pass

200개, use_debate=True(caveat 수정 후), max_workers=8: 310.7초
에러: 0건
success: 131
stuck: 43
no_known_fix: 26

--- 규칙 기반 ---
success: 148
stuck: 27
no_known_fix: 25

총 19/200건 차이
[success -> no_known_fix] O=C1OC(CN2CCOCC2)CN1N=Cc1ccc([N+](=O)[O-])o1
[success -> stuck] OCCOCCOCCOCCO
[no_known_fix -> stuck] ClC1=C(Cl)[C@]2(Cl)[C@@H]3[C@@H]4C[C@H]([C@@H]3[C@@]1(Cl)C2(Cl)Cl)[C@H]1O[C@@H]41
[success -> stuck] Nc1ccc(C(=O)[O-])c(O)c1
[success -> stuck] CCOP(=S)(OCC)SCSC(C)(C)C
[success -> stuck] OCCSCSCCO
[success -> stuck] O=[N+]([O-])c1ccc(Cl)c([N+](=O)[O-])c1
[success -> no_known_fix] O=C([O-])CCC(=O)OC[C@@H](NC(=O)C(Cl)Cl)[C@H](O)c1ccc([N+](=O)[O-])cc1
[success -> stuck] CN1CSC(=S)N(C)C1
[success -> stuck] O=CC=C(c1ccccc1)c1ccccc1
[success -> stuck] Nc1ccc2ccccc2c1
[success -> no_known_fix] Cc1ccc(/C=N/n2c(-c3ccccc3)csc2=S)cc1
[success -> stuck] Nc1cc(Cl)c(NC2=NCCN2)c(Cl)c1
[no_known_fix -> stuck] Nc1ccc(/N=N\c2ccccc2)c(N)c1
[success -> stuck] CCCCNC(=O)OCC#CI
[success -> stuck] CC[N+](=O)[O-

[06:43:12] Incomplete atom labelling, cannot make bond
[06:43:12] Incomplete atom labelling, cannot make bond


In [7]:
ma1_verdicts = Counter()
for smi, r in results_debate_v2:
    for h in r['history']:
        for attempt in (h.get('debate_rounds') or []):
            if attempt['rule'] == 'Michael_acceptor_1':
                ma1_verdicts[attempt['verdict']] += 1

print("Michael_acceptor_1 토의 판정 분포 (caveat 수정 후):", dict(ma1_verdicts))

Michael_acceptor_1 토의 판정 분포 (caveat 수정 후): {'approved': 3}


In [ ]:
!cat src/tools/molecule_editor.py

In [10]:
%%writefile tests/test_regression.py
"""회귀 방지 테스트 스위트. 반복적으로 겪은 문제들
(함수 중복 정의, 수정이 실제로 반영 안 됨, 핵심 파이프라인 깨짐)을
매 세션 시작 시 한 번에 잡아내기 위한 것. 네트워크/API 호출 없는
테스트만 포함(빠르게, 매번 돌릴 수 있게)."""

import ast
import glob
import os
import pytest

def pytest_configure(config):
    config.addinivalue_line("markers", "slow: requires network/LLM API access")

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def _all_source_files():
    patterns = ["src/tools/*.py", "src/models/*.py", "models/*.py"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(REPO_ROOT, pat)))
    return files


def test_no_duplicate_top_level_function_definitions():
    """%%writefile -a로 같은 함수를 두 번 추가해서, 나중(옛날) 버전이
    최종 반영되는 사고(batch_iterative_fix_loop 사례)를 방지."""
    problems = []
    for path in _all_source_files():
        with open(path) as f:
            source = f.read()
        tree = ast.parse(source)
        names = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
        seen = set()
        for name in names:
            if name in seen:
                problems.append(f"{path}: '{name}' 중복 정의")
            seen.add(name)
    assert not problems, "\n".join(problems)


def test_all_source_files_parse():
    """모든 소스 파일이 문법적으로 유효한지."""
    problems = []
    for path in _all_source_files():
        with open(path) as f:
            try:
                ast.parse(f.read())
            except SyntaxError as e:
                problems.append(f"{path}: {e}")
    assert not problems, "\n".join(problems)


def test_precedent_library_structure():
    from src.tools.precedent_library import PRECEDENT_LIBRARY
    assert len(PRECEDENT_LIBRARY) >= 24, f"선례 수가 예상보다 적음: {len(PRECEDENT_LIBRARY)}"
    required_keys = {"rule", "type", "description"}
    for i, p in enumerate(PRECEDENT_LIBRARY):
        missing = required_keys - p.keys()
        assert not missing, f"{i}번째 항목에 키 누락: {missing}"


def test_docking_targets_have_caveat_field():
    """caveat 필드 누락 회귀 방지 (오늘 겪은 EGFR 편향 사고)."""
    from src.tools.docking import DOCKING_TARGETS
    for rule, info in DOCKING_TARGETS.items():
        assert "caveat" in info, f"{rule}에 caveat 필드 없음"


def test_agent_required_functions_exist():
    """agent.py에 있어야 할 핵심 함수/설정 함수들이 다 있는지."""
    import src.tools.agent as agent
    required = [
        "ask_llm_which_problem_to_fix", "ask_llm_which_candidate_to_use",
        "ask_llm_debate_fix", "should_debate",
        "set_debate_budget", "set_sascorer_module", "set_tox_predictor",
        "_try_get_docking_evidence", "_try_compute_score", "_try_get_activity_risk",
    ]
    missing = [name for name in required if not hasattr(agent, name)]
    assert not missing, f"agent.py에 없는 함수: {missing}"


def test_batch_iterative_fix_loop_signature_has_debate_params():
    import inspect
    from src.tools.molecule_editor import batch_iterative_fix_loop
    sig = inspect.signature(batch_iterative_fix_loop)
    assert "use_debate" in sig.parameters
    assert "debate_max_rounds" in sig.parameters


def test_molecule_editor_debate_rounds_recorded_in_history():
    """감사추적용 debate_rounds 키가 iterative_fix_loop 코드에 존재하는지
    (실제 토의 왕복 기록 여부는 별도 통합테스트에서 확인)."""
    with open(os.path.join(REPO_ROOT, "src/tools/molecule_editor.py")) as f:
        content = f.read()
    assert "debate_rounds" in content


@pytest.mark.slow
def test_smoke_iterative_fix_loop_resolves_simple_chain():
    """규칙 기반(LLM 없음) 스모크 테스트: 가장 기본적인 회귀 방지."""
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    clear_failure_memory()
    r = iterative_fix_loop("CCCCCCCCCCCCCCCC", max_iterations=10, candidate_idx=0)
    assert r["status"] == "success"


@pytest.mark.slow
def test_smoke_conflict_resolution_picks_lower_remaining_count():
    """다중문제 충돌조정이 실제로 남는 문제 수 적은 쪽을 먼저 고르는지."""
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    clear_failure_memory()
    smi = "NNC(=O)CP(=O)(c1ccccc1)c1ccccc1"  # hydrazine + phosphor
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    step1 = r["history"][1]
    assert step1.get("fixed_rule") == "hydrazine"
    assert "충돌 조정" in step1.get("problem_reason", "")

@pytest.mark.slow
def test_integration_debate_resolves_itaconic_acid_case():
    """어제 caveat 버그의 원인이었던 실제 케이스가 여전히 정상 동작하는지.
    (client_qwen, QWEN_MODEL은 노트북 전역에 이미 정의돼 있어야 함)"""
    import builtins
    client_qwen = getattr(builtins, "client_qwen", None) or globals().get("client_qwen")
    if client_qwen is None:
        pytest.skip("client_qwen이 정의 안 됨 (노트북 셀에서 미리 만들어야 함)")

    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    from src.tools.agent import _try_get_docking_evidence

    clear_failure_memory()
    r = iterative_fix_loop(
        "C=C(CC(=O)O)C(=O)O", max_iterations=10, candidate_idx=0,
        llm_client=client_qwen, llm_model="qwen3.8-max", llm_client_type="openai_compatible",
        use_debate=True, debate_max_rounds=2,
    )
    assert r["status"] in ("success", "stuck")
    step1 = r["history"][1] if len(r["history"]) > 1 else None
    assert step1 is not None
    debate_attempts = step1.get("debate_rounds") or []
    assert len(debate_attempts) > 0, "토의가 아예 발동 안 함 (should_debate 트리거 확인 필요)"


@pytest.mark.slow
def test_integration_docking_evidence_reaches_debate_prompt():
    """도킹이 실제로 critic 프롬프트에 삽입되는지 end-to-end 확인."""
    client_qwen = globals().get("client_qwen")
    if client_qwen is None:
        pytest.skip("client_qwen이 정의 안 됨")

    from src.tools.agent import ask_llm_debate_fix
    from src.tools.molecule_editor import propose_fix
    from src.tools.replacement_library import get_replacement_candidates

    info = get_replacement_candidates("catechol")
    candidate = info["candidates"][0]
    trial = propose_fix("NCCc1ccc(O)c(O)c1", "catechol", 0)
    assert trial is not None

    result = ask_llm_debate_fix(
        client_qwen, "qwen3.8-max", "NCCc1ccc(O)c(O)c1", trial["new_smiles"], "catechol",
        candidate["name"], candidate.get("rationale", ""), client_type="openai_compatible",
    )
    assert result["final_verdict"] in ("approved", "rejected", "escalate")

Overwriting tests/test_regression.py


In [11]:
import ast
with open('tests/test_regression.py') as f:
    ast.parse(f.read())
print("✅ 문법 정상")

!cd /content/laidd-2026 && python -m pytest tests/test_regression.py -v -m "not slow"

✅ 문법 정상
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/laidd-2026
plugins: langsmith-0.11.0, anyio-4.14.2, typeguard-4.6.0
collected 11 items / 4 deselected / 7 selected                                 

tests/test_regression.py::test_no_duplicate_top_level_function_definitions PASSED [ 14%]
tests/test_regression.py::test_all_source_files_parse PASSED             [ 28%]
tests/test_regression.py::test_precedent_library_structure PASSED        [ 42%]
tests/test_regression.py::test_docking_targets_have_caveat_field PASSED  [ 57%]
tests/test_regression.py::test_agent_required_functions_exist PASSED     [ 71%]
tests/test_regression.py::test_batch_iterative_fix_loop_signature_has_debate_params PASSED [ 85%]
tests/test_regression.py::test_molecule_editor_debate_rounds_recorded_in_history PASSED [100%]

=============================== warn

In [12]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add slow integration test stubs (Qwen client-dependent, skipped outside notebook context) to test_regression.py" && git push

[main 068a7e6] Add slow integration test stubs (Qwen client-dependent, skipped outside notebook context) to test_regression.py
 6 files changed, 99 insertions(+), 40 deletions(-)
Enumerating objects: 20, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 2 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (11/11), 2.71 KiB | 1.35 MiB/s, done.
Total 11 (delta 9), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (9/9), completed with 8 local objects.
To https://github.com/Dec32th/laidd-2026.git
   145835b..068a7e6  main -> main


In [13]:
smi = "CCOP(=S)(OCC)SCSC(C)(C)C"

problems = detect_toxicophores(smi)
print("진단된 문제:", [p['rule_name'] for p in problems])
known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
print("known(라이브러리 등록된) 문제:", [p['rule_name'] for p in known])

# 규칙 기반(토의 없음) 이력
clear_failure_memory()
r_rule = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
print("\n=== 규칙 기반 ===")
print(generate_audit_report(r_rule, original_smiles=smi))

# 토의 포함 이력
clear_failure_memory()
r_debate = iterative_fix_loop(
    smi, max_iterations=10, candidate_idx=0,
    llm_client=client_qwen, llm_model=QWEN_MODEL, llm_client_type="openai_compatible",
    use_debate=True, debate_max_rounds=2,
)
print("\n=== 토의 포함 ===")
print(generate_audit_report(r_debate, original_smiles=smi))

진단된 문제: ['het-C-het_not_in_ring', 'phosphor']
known(라이브러리 등록된) 문제: ['het-C-het_not_in_ring', 'phosphor']

=== 규칙 기반 ===
치환 감사추적 리포트
원본 분자: CCOP(=S)(OCC)SCSC(C)(C)C
최종 상태: success
최종 분자: C=O

--- 단계별 이력 ---

[스텝 0] CCOP(=S)(OCC)SCSC(C)(C)C
  진단된 문제: ['het-C-het_not_in_ring', 'phosphor']

[스텝 1] C=S
  진단된 문제: ['Thiocarbonyl_group']
  고친 규칙: het-C-het_not_in_ring (판단 근거: 규칙 기반(충돌 조정: 남는 문제 수 적은 순 - {'het-C-het_not_in_ring': 1, 'phosphor': 999}))
  적용된 치환: ketone/ester (one heteroatom substituent removed, C=O formed)
  candidate 선택 근거: 규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도) (candidate_idx=0, 완전 해소)

[스텝 2] C=O
  고친 규칙: Thiocarbonyl_group (판단 근거: 규칙 기반(충돌 조정: 남는 문제 수 적은 순 - {'Thiocarbonyl_group': 0}))
  적용된 치환: carbonyl (O replacing S)
  candidate 선택 근거: 규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도) (candidate_idx=0, 완전 해소)

=== 토의 포함 ===
치환 감사추적 리포트
원본 분자: CCOP(=S)(OCC)SCSC(C)(C)C
최종 상태: stuck
최종 분자: C=S

--- 단계별 이력 ---

[스텝 0] CCOP(=S)(OCC)SCSC(C)(C)C
  진단된 문제: ['het-

In [14]:
import random
from rdkit import Chem

random.seed(7)
sample_check = random.sample(list(data['smiles_valid']), 300)

destructive_cases = []

for smi in sample_check:
    mol_before = Chem.MolFromSmiles(smi)
    if mol_before is None:
        continue
    atoms_before = mol_before.GetNumHeavyAtoms()

    clear_failure_memory()
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)

    if r['status'] != 'success':
        continue

    mol_after = Chem.MolFromSmiles(r['final_smiles'])
    if mol_after is None:
        continue
    atoms_after = mol_after.GetNumHeavyAtoms()

    loss_ratio = 1 - (atoms_after / atoms_before) if atoms_before > 0 else 0
    if loss_ratio >= 0.5:  # 무거운 원자 절반 이상 손실
        destructive_cases.append({
            "original": smi, "final": r['final_smiles'],
            "atoms_before": atoms_before, "atoms_after": atoms_after,
            "loss_ratio": loss_ratio,
        })

print(f"success 처리된 것 중 원자 50% 이상 손실된 케이스: {len(destructive_cases)}건 / 샘플 300개")
for c in destructive_cases[:15]:
    print(f"  {c['original']} ({c['atoms_before']}원자) -> {c['final']} ({c['atoms_after']}원자, {c['loss_ratio']:.0%} 손실)")

[06:50:16] Incomplete atom labelling, cannot make bond
[06:50:16] Incomplete atom labelling, cannot make bond


success 처리된 것 중 원자 50% 이상 손실된 케이스: 8건 / 샘플 300개
  CN=C=O (4원자) -> CN (2원자, 50% 손실)
  C[N+](C)(C)CC(O)CCl (9원자) -> CN(C)C (4원자, 56% 손실)
  CCCCCCCCCCCCCCCCCCOP(=O)([O-])OC1CC[N+](C)(C)CC1 (31원자) -> CNC (3원자, 90% 손실)
  CCOP(=S)(OCC)SCSC(C)(C)C (15원자) -> C=O (2원자, 87% 손실)
  OCCSCSCCO (9원자) -> C=O (2원자, 78% 손실)
  CC(=O)O[C@H]1C[C@@H]2CC[C@@H]3[C@H](CC[C@@]4(C)[C@H]3C[C@H](N3CC[N+](C)(C)CC3)[C@@H]4OC(C)=O)[C@@]2(C)C[C@@H]1N1CC[N+](C)(C)CC1 (43원자) -> CNC (3원자, 93% 손실)
  Cc1cc(N)c2ccccc2[n+]1CCCCCCCCCC[n+]1c(C)cc(N)c2ccccc21 (34원자) -> Cc1cc(N)c2ccccc2n1 (12원자, 65% 손실)
  CC(=O)NCCCS(=O)(=O)[O-].CC(=O)NCCCS(=O)(=O)[O-] (22원자) -> CC(=O)NCCS(N)(=O)=O (10원자, 55% 손실)


In [19]:
import ast
with open('src/tools/molecule_editor.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ 문법 정상")
print("✅ 파괴적 편집 가드 반영:", '파괴적 편집' in content)

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory

clear_failure_memory()
r = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert r['status'] == 'success'
print("✅ 회귀 없음 확인")

destructive_smiles = [
    "CN=C=O",
    "C[N+](C)(C)CC(O)CCl",
    "CCCCCCCCCCCCCCCCCCOP(=O)([O-])OC1CC[N+](C)(C)CC1",
    "CCOP(=S)(OCC)SCSC(C)(C)C",
    "OCCSCSCCO",
    "CC(=O)O[C@H]1C[C@@H]2CC[C@@H]3[C@H](CC[C@@]4(C)[C@H]3C[C@H](N3CC[N+](C)(C)CC3)[C@@H]4OC(C)=O)[C@@]2(C)C[C@@H]1N1CC[N+](C)(C)CC1",
    "Cc1cc(N)c2ccccc2[n+]1CCCCCCCCCC[n+]1c(C)cc(N)c2ccccc21",
    "CC(=O)NCCCS(=O)(=O)[O-].CC(=O)NCCCS(=O)(=O)[O-]",
]

for smi in destructive_smiles:
    clear_failure_memory()
    r2 = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    mol_before = Chem.MolFromSmiles(smi)
    mol_after = Chem.MolFromSmiles(r2['final_smiles']) if r2.get('final_smiles') else None
    atoms_before = mol_before.GetNumHeavyAtoms() if mol_before else None
    atoms_after = mol_after.GetNumHeavyAtoms() if mol_after else None
    print(f"{smi[:40]}... -> status={r2['status']}, {atoms_before}->{atoms_after}원자")

✅ 문법 정상
✅ 파괴적 편집 가드 반영: True
✅ 회귀 없음 확인
CN=C=O... -> status=stuck, 4->4원자
C[N+](C)(C)CC(O)CCl... -> status=stuck, 9->9원자
CCCCCCCCCCCCCCCCCCOP(=O)([O-])OC1CC[N+](... -> status=stuck, 31->35원자
CCOP(=S)(OCC)SCSC(C)(C)C... -> status=stuck, 15->15원자
OCCSCSCCO... -> status=stuck, 9->9원자
CC(=O)O[C@H]1C[C@@H]2CC[C@@H]3[C@H](CC[C... -> status=stuck, 43->43원자
Cc1cc(N)c2ccccc2[n+]1CCCCCCCCCC[n+]1c(C)... -> status=stuck, 34->36원자
CC(=O)NCCCS(=O)(=O)[O-].CC(=O)NCCCS(=O)(... -> status=stuck, 22->22원자


[06:58:13] Incomplete atom labelling, cannot make bond
[06:58:13] Incomplete atom labelling, cannot make bond
[06:58:13] Incomplete atom labelling, cannot make bond


In [20]:
random.seed(7)
sample_check2 = random.sample(list(data['smiles_valid']), 300)

destructive_cases2 = []
status_counter_guard = Counter()

for smi in sample_check2:
    mol_before = Chem.MolFromSmiles(smi)
    if mol_before is None:
        continue
    atoms_before = mol_before.GetNumHeavyAtoms()

    clear_failure_memory()
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status_counter_guard[r['status']] += 1

    if r['status'] != 'success':
        continue

    mol_after = Chem.MolFromSmiles(r['final_smiles'])
    if mol_after is None:
        continue
    atoms_after = mol_after.GetNumHeavyAtoms()
    loss_ratio = 1 - (atoms_after / atoms_before) if atoms_before > 0 else 0
    if loss_ratio >= 0.5:
        destructive_cases2.append((smi, r['final_smiles'], loss_ratio))

print("가드 적용 후 상태 분포:", dict(status_counter_guard))
print(f"\n가드 적용 후에도 남은 파괴적 케이스: {len(destructive_cases2)}건")
for c in destructive_cases2:
    print(c)

[06:59:22] Incomplete atom labelling, cannot make bond
[06:59:22] Incomplete atom labelling, cannot make bond
[06:59:22] Incomplete atom labelling, cannot make bond


가드 적용 후 상태 분포: {'stuck': 44, 'success': 214, 'no_known_fix': 42}

가드 적용 후에도 남은 파괴적 케이스: 0건


In [6]:
import importlib, json, ast, time, os, random
from collections import Counter
from rdkit import Chem
import requests

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.precedent_library
import src.tools.agent
import src.tools.docking
import src.tools.activity_metrics
import src.tools.audit
import models.tox_baseline

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, batch_iterative_fix_loop, clear_failure_memory
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents
from src.tools.docking import auto_dock_precedent, inspect_hetatm, DOCKING_TARGETS
from src.tools.activity_metrics import compute_activity_preservation_metrics, classify_activity_risk_v3
from src.tools.audit import generate_audit_report
from models.tox_baseline import train_tox21_baseline, make_tox_predictor

data = load_tox21_clean(random_state=7)

base_url = "https://www.guidetopharmacology.org/services"
def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()
def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina_bin
!chmod +x vina_bin

import urllib.request, sys
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py", "sascorer.py")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz", "fpscores.pkl.gz")
sys.path.append('.')
import sascorer

from src.tools.agent import set_sascorer_module, set_tox_predictor
set_sascorer_module(sascorer)
tox_model = train_tox21_baseline(data)
set_tox_predictor(make_tox_predictor(tox_model))

print(f"선례 수: {len(PRECEDENT_LIBRARY)} (24여야 정상)")
print("vina_bin 존재:", os.path.exists('vina_bin'))
print("caveat 필드 반영 여부:", 'caveat' in open('src/tools/docking.py').read())
print("all_debate_logs_for_step 반영 여부:", 'all_debate_logs_for_step' in open('src/tools/molecule_editor.py').read())
print("파괴적 편집 가드 반영 여부(아직 커밋 안 됨, False 예상):", '파괴적 편집' in open('src/tools/molecule_editor.py').read())

[13:08:45] WARNING: not removing hydrogen atom without neighbors
[13:08:45] Explicit valence for atom # 8 Al, 6, is greater than permitted
[13:08:46] Explicit valence for atom # 3 Al, 6, is greater than permitted
[13:08:46] Explicit valence for atom # 4 Al, 6, is greater than permitted
[13:08:46] Explicit valence for atom # 4 Al, 6, is greater than permitted
[13:08:46] Explicit valence for atom # 9 Al, 6, is greater than permitted
[13:08:46] Explicit valence for atom # 5 Al, 6, is greater than permitted
[13:08:46] Explicit valence for atom # 16 Al, 6, is greater than permitted
[13:08:47] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[13:08:47] WARNING: not removing hydrogen atom without neighbors


선례 수: 24 (24여야 정상)
vina_bin 존재: True
caveat 필드 반영 여부: True
all_debate_logs_for_step 반영 여부: True
파괴적 편집 가드 반영 여부(아직 커밋 안 됨, False 예상): False


In [8]:
%%writefile src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib
from src.tools.agent import (ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use,
                                  ask_llm_debate_fix, should_debate)

def _library_version_hash():
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]
            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue
            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue
            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')
            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def _candidate_order_for_rule(rule_name: str, preferred_idx: int):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return [preferred_idx]
    n = len(info['candidates'])
    order = [preferred_idx] if 0 <= preferred_idx < n else []
    order += [i for i in range(n) if i != preferred_idx]
    return order


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini",
                        use_failure_memory: bool = True, use_debate: bool = False,
                        debate_max_rounds: int = 2):
    """진단->치환->재평가를 반복.
    핵심: candidate가 '화학적으로 유효(is_valid)'해도 대상 규칙이 실제로
    해소됐는지 재진단(detect_toxicophores)까지 확인한다. 그렇지 않으면
    항상 valid하지만 문제를 안 고치는 candidate(예: 단순 삽입형)가
    무한 반복 채택되어 진짜 해법(예: 분기형)으로 넘어가지 못하는 문제가
    있었음. 완전 해소가 안 되면 마지막으로 시도한(=대개 더 나은)
    valid 결과를 fallback으로 채택해 다음 iteration에서 계속 개선."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            # 다중 문제 충돌 조정: 각 문제를 먼저 고쳤을 때 남는 전체
            # toxicophore 수가 가장 적어지는 순서로 정렬(그리디, LLM 미사용).
            sim_scores = {}
            for p in known_problems:
                rn = p['rule_name']
                try:
                    trial = propose_fix(current, rn, candidate_idx)
                    if trial is None or not trial.get('is_valid'):
                        sim_scores[rn] = 999
                        continue
                    remaining = detect_toxicophores(trial['new_smiles'])
                    sim_scores[rn] = len(remaining)
                except Exception:
                    sim_scores[rn] = 999
            ordered_rules = sorted(sim_scores, key=sim_scores.get)
            problem_reason = f"규칙 기반(충돌 조정: 남는 문제 수 적은 순 - {sim_scores})"

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            all_debate_logs_for_step = []
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                preferred_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if preferred_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                preferred_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도)"

            rule_fixed = None
            fallback_attempt = None
            fallback_used_idx = None
            fallback_reason = None

            for try_idx in _candidate_order_for_rule(candidate_rule, preferred_candidate_idx):
                memory_key = (current, candidate_rule, try_idx, _library_version_hash())
                if use_failure_memory and memory_key in _FAILURE_MEMORY:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](memory-skip)")
                    continue

                attempt = propose_fix(current, candidate_rule, try_idx)
                if attempt is None or not attempt.get('is_valid'):
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}]")
                    if use_failure_memory:
                        _FAILURE_MEMORY[memory_key] = True
                    continue

                # 파괴적 편집 가드: 무거운 원자 50% 이상이 사라지면 "고침"이
                # 아니라 분자 자체를 파괴한 것으로 간주(remove_substituent류
                # 편집이 비고리 분자의 대부분을 통째로 잘라내는 사고 방지)
                mol_current_check = Chem.MolFromSmiles(current)
                mol_new_check = Chem.MolFromSmiles(attempt['new_smiles'])
                if mol_current_check and mol_new_check:
                    atoms_before = mol_current_check.GetNumHeavyAtoms()
                    atoms_after = mol_new_check.GetNumHeavyAtoms()
                    loss_ratio = 1 - (atoms_after / atoms_before) if atoms_before > 0 else 0
                    if loss_ratio >= 0.5:
                        failed_attempts.append(
                            f"{candidate_rule}[idx={try_idx}](파괴적 편집 거부: 원자 {loss_ratio:.0%} 손실)"
                        )
                        if use_failure_memory:
                            _FAILURE_MEMORY[memory_key] = True
                        continue

                # valid해도 실제로 이 규칙이 재진단에서 사라졌는지 확인
                recheck = detect_toxicophores(attempt['new_smiles'])
                still_flagged = any(p['rule_name'] == candidate_rule for p in recheck)

                if not still_flagged:
                    candidate_obj = get_replacement_candidates(candidate_rule)['candidates'][try_idx]
                    debate_suffix = ""

                    if use_debate and llm_client is not None and should_debate(candidate_obj.get('rationale', '')):
                        debate_result = ask_llm_debate_fix(
                            llm_client, llm_model, current, attempt['new_smiles'], candidate_rule,
                            candidate_obj['name'], candidate_obj.get('rationale', ''),
                            client_type=llm_client_type, max_rounds=debate_max_rounds,
                        )
                        if debate_result['final_verdict'] == 'rejected':
                            all_debate_logs_for_step.append({
                                "rule": candidate_rule, "candidate_idx": try_idx,
                                "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                            })
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 결과 반려)")
                            if use_failure_memory:
                                _FAILURE_MEMORY[memory_key] = True
                            continue
                        elif debate_result['final_verdict'] == 'escalate':
                            all_debate_logs_for_step.append({
                                "rule": candidate_rule, "candidate_idx": try_idx,
                                "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                            })
                            flagged_for_review.add(candidate_rule)
                            if candidate_rule not in skipped_rules:
                                skipped_rules.append(candidate_rule)
                            skipped_details.append({
                                "rule_name": candidate_rule,
                                "reason": f"LLM 토의가 {debate_max_rounds}라운드 안에 합의에 도달하지 못해 "
                                          f"사람 검토로 넘김 (마지막 논쟁: {debate_result['rounds'][-1]['text']})",
                                "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                            })
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 합의 실패, escalate)")
                            all_debate_logs_for_step.append({
                                "rule": candidate_rule, "candidate_idx": try_idx,
                                "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                            })
                            continue
                        all_debate_logs_for_step.append({
                            "rule": candidate_rule, "candidate_idx": try_idx,
                            "verdict": debate_result['final_verdict'], "rounds": debate_result['rounds'],
                        })
                        debate_suffix = " (토의 승인)"

                    rule_fixed = attempt
                    candidate_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 완전 해소){debate_suffix}"
                    break
                else:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](valid이나 미해소)")
                    fallback_attempt = attempt
                    fallback_used_idx = try_idx
                    fallback_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 부분 개선/다음 iteration에서 계속)"

            if rule_fixed is None and fallback_attempt is not None:
                rule_fixed = fallback_attempt
                candidate_reason = fallback_reason

            if rule_fixed is not None:
                fixed = rule_fixed
                target_rule = candidate_rule
                break

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 "
                              f"({failed_attempts}) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 "
                              f"실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 "
                              f"등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙/candidate {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
            "debate_rounds": all_debate_logs_for_step,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}


def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True, use_debate=False,
                               debate_max_rounds=2):
    """여러 분자에 iterative_fix_loop를 스레드 병렬로 적용.
    LLM API 호출이 병목인 경우(네트워크 대기 시간) 유효한 개선이며,
    화학 계산 로직(iterative_fix_loop 자체)은 전혀 수정하지 않는다.
    반환: [(smiles, result_dict), ...] (완료 순서, 입력 순서와 다를 수 있음)
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def _process_one(smi):
        r = iterative_fix_loop(
            smi, max_iterations=max_iterations, candidate_idx=candidate_idx,
            llm_client=llm_client, llm_model=llm_model, llm_client_type=llm_client_type,
            use_debate=use_debate, debate_max_rounds=debate_max_rounds,
        )
        return smi, r

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_process_one, smi): smi for smi in smiles_list}
        for i, future in enumerate(as_completed(futures)):
            smi, r = future.result()
            results.append((smi, r))
            if progress:
                print(f"[{i+1}/{len(smiles_list)}] {smi[:30]} -> {r['status']}")
    return results




Overwriting src/tools/molecule_editor.py


In [10]:
%%writefile tests/test_regression.py
"""회귀 방지 테스트 스위트. 반복적으로 겪은 문제들
(함수 중복 정의, 수정이 실제로 반영 안 됨, 핵심 파이프라인 깨짐)을
매 세션 시작 시 한 번에 잡아내기 위한 것. 네트워크/API 호출 없는
테스트만 포함(빠르게, 매번 돌릴 수 있게)."""

import ast
import glob
import os
import pytest

def pytest_configure(config):
    config.addinivalue_line("markers", "slow: requires network/LLM API access")

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def _all_source_files():
    patterns = ["src/tools/*.py", "src/models/*.py", "models/*.py"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(REPO_ROOT, pat)))
    return files


def test_no_duplicate_top_level_function_definitions():
    """%%writefile -a로 같은 함수를 두 번 추가해서, 나중(옛날) 버전이
    최종 반영되는 사고(batch_iterative_fix_loop 사례)를 방지."""
    problems = []
    for path in _all_source_files():
        with open(path) as f:
            source = f.read()
        tree = ast.parse(source)
        names = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
        seen = set()
        for name in names:
            if name in seen:
                problems.append(f"{path}: '{name}' 중복 정의")
            seen.add(name)
    assert not problems, "\n".join(problems)


def test_all_source_files_parse():
    """모든 소스 파일이 문법적으로 유효한지."""
    problems = []
    for path in _all_source_files():
        with open(path) as f:
            try:
                ast.parse(f.read())
            except SyntaxError as e:
                problems.append(f"{path}: {e}")
    assert not problems, "\n".join(problems)


def test_precedent_library_structure():
    from src.tools.precedent_library import PRECEDENT_LIBRARY
    assert len(PRECEDENT_LIBRARY) >= 24, f"선례 수가 예상보다 적음: {len(PRECEDENT_LIBRARY)}"
    required_keys = {"rule", "type", "description"}
    for i, p in enumerate(PRECEDENT_LIBRARY):
        missing = required_keys - p.keys()
        assert not missing, f"{i}번째 항목에 키 누락: {missing}"


def test_docking_targets_have_caveat_field():
    """caveat 필드 누락 회귀 방지 (오늘 겪은 EGFR 편향 사고)."""
    from src.tools.docking import DOCKING_TARGETS
    for rule, info in DOCKING_TARGETS.items():
        assert "caveat" in info, f"{rule}에 caveat 필드 없음"


def test_agent_required_functions_exist():
    """agent.py에 있어야 할 핵심 함수/설정 함수들이 다 있는지."""
    import src.tools.agent as agent
    required = [
        "ask_llm_which_problem_to_fix", "ask_llm_which_candidate_to_use",
        "ask_llm_debate_fix", "should_debate",
        "set_debate_budget", "set_sascorer_module", "set_tox_predictor",
        "_try_get_docking_evidence", "_try_compute_score", "_try_get_activity_risk",
    ]
    missing = [name for name in required if not hasattr(agent, name)]
    assert not missing, f"agent.py에 없는 함수: {missing}"


def test_batch_iterative_fix_loop_signature_has_debate_params():
    import inspect
    from src.tools.molecule_editor import batch_iterative_fix_loop
    sig = inspect.signature(batch_iterative_fix_loop)
    assert "use_debate" in sig.parameters
    assert "debate_max_rounds" in sig.parameters


def test_molecule_editor_debate_rounds_recorded_in_history():
    """감사추적용 debate_rounds 키가 iterative_fix_loop 코드에 존재하는지
    (실제 토의 왕복 기록 여부는 별도 통합테스트에서 확인)."""
    with open(os.path.join(REPO_ROOT, "src/tools/molecule_editor.py")) as f:
        content = f.read()
    assert "debate_rounds" in content


@pytest.mark.slow
def test_smoke_iterative_fix_loop_resolves_simple_chain():
    """규칙 기반(LLM 없음) 스모크 테스트: 가장 기본적인 회귀 방지."""
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    clear_failure_memory()
    r = iterative_fix_loop("CCCCCCCCCCCCCCCC", max_iterations=10, candidate_idx=0)
    assert r["status"] == "success"


@pytest.mark.slow
def test_smoke_conflict_resolution_picks_lower_remaining_count():
    """다중문제 충돌조정이 실제로 남는 문제 수 적은 쪽을 먼저 고르는지."""
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    clear_failure_memory()
    smi = "NNC(=O)CP(=O)(c1ccccc1)c1ccccc1"  # hydrazine + phosphor
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    step1 = r["history"][1]
    assert step1.get("fixed_rule") == "hydrazine"
    assert "충돌 조정" in step1.get("problem_reason", "")

@pytest.mark.slow
def test_integration_debate_resolves_itaconic_acid_case():
    """어제 caveat 버그의 원인이었던 실제 케이스가 여전히 정상 동작하는지.
    (client_qwen, QWEN_MODEL은 노트북 전역에 이미 정의돼 있어야 함)"""
    import builtins
    client_qwen = getattr(builtins, "client_qwen", None) or globals().get("client_qwen")
    if client_qwen is None:
        pytest.skip("client_qwen이 정의 안 됨 (노트북 셀에서 미리 만들어야 함)")

    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    from src.tools.agent import _try_get_docking_evidence

    clear_failure_memory()
    r = iterative_fix_loop(
        "C=C(CC(=O)O)C(=O)O", max_iterations=10, candidate_idx=0,
        llm_client=client_qwen, llm_model="qwen3.8-max", llm_client_type="openai_compatible",
        use_debate=True, debate_max_rounds=2,
    )
    assert r["status"] in ("success", "stuck")
    step1 = r["history"][1] if len(r["history"]) > 1 else None
    assert step1 is not None
    debate_attempts = step1.get("debate_rounds") or []
    assert len(debate_attempts) > 0, "토의가 아예 발동 안 함 (should_debate 트리거 확인 필요)"


@pytest.mark.slow
def test_integration_docking_evidence_reaches_debate_prompt():
    """도킹이 실제로 critic 프롬프트에 삽입되는지 end-to-end 확인."""
    client_qwen = globals().get("client_qwen")
    if client_qwen is None:
        pytest.skip("client_qwen이 정의 안 됨")

    from src.tools.agent import ask_llm_debate_fix
    from src.tools.molecule_editor import propose_fix
    from src.tools.replacement_library import get_replacement_candidates

    info = get_replacement_candidates("catechol")
    candidate = info["candidates"][0]
    trial = propose_fix("NCCc1ccc(O)c(O)c1", "catechol", 0)
    assert trial is not None

    result = ask_llm_debate_fix(
        client_qwen, "qwen3.8-max", "NCCc1ccc(O)c(O)c1", trial["new_smiles"], "catechol",
        candidate["name"], candidate.get("rationale", ""), client_type="openai_compatible",
    )
    assert result["final_verdict"] in ("approved", "rejected", "escalate")

    def test_destructive_edit_guard_present():
    with open(os.path.join(REPO_ROOT, "src/tools/molecule_editor.py")) as f:
        content = f.read()
    assert "파괴적 편집" in content
    assert "loss_ratio" in content


def test_destructive_edit_guard_rejects_thiophosphate_case():
    """실제로 파괴적 편집(원자 대부분 삭제)을 막는지 회귀 검증."""
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    from rdkit import Chem

    smi = "CCOP(=S)(OCC)SCSC(C)(C)C"
    clear_failure_memory()
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)

    mol_before = Chem.MolFromSmiles(smi)
    mol_after = Chem.MolFromSmiles(r["final_smiles"]) if r.get("final_smiles") else None
    if mol_before and mol_after:
        atoms_before = mol_before.GetNumHeavyAtoms()
        atoms_after = mol_after.GetNumHeavyAtoms()
        loss_ratio = 1 - (atoms_after / atoms_before) if atoms_before > 0 else 0
        assert loss_ratio < 0.5, f"파괴적 편집 가드가 뚫림: {loss_ratio:.0%} 원자 손실"


Overwriting tests/test_regression.py


In [13]:
with open('tests/test_regression.py') as f:
    content = f.read()

old = '\n    def test_destructive_edit_guard_present():\n    with open(os.path.join(REPO_ROOT, "src/tools/molecule_editor.py")) as f:\n        content = f.read()\n    assert "파괴적 편집" in content\n    assert "loss_ratio" in content\n'
new = '\n\ndef test_destructive_edit_guard_present():\n    with open(os.path.join(REPO_ROOT, "src/tools/molecule_editor.py")) as f:\n        content = f.read()\n    assert "파괴적 편집" in content\n    assert "loss_ratio" in content\n'

assert content.count(old) == 1, f"매치 개수: {content.count(old)} (1이어야 함)"
content = content.replace(old, new)

with open('tests/test_regression.py', 'w') as f:
    f.write(content)

import ast
ast.parse(content)
print("✅ 수정 완료, 문법 정상")

✅ 수정 완료, 문법 정상


In [14]:
import ast
with open('src/tools/molecule_editor.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ molecule_editor.py 문법 정상, 가드 반영:", '파괴적 편집' in content)

with open('tests/test_regression.py') as f:
    ast.parse(f.read())
print("✅ test_regression.py 문법 정상")

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory

clear_failure_memory()
r = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert r['status'] == 'success'
print("✅ 회귀 없음 확인")

!cd /content/laidd-2026 && python -m pytest tests/test_regression.py -v -m "not slow"

✅ molecule_editor.py 문법 정상, 가드 반영: True
✅ test_regression.py 문법 정상
✅ 회귀 없음 확인
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/laidd-2026
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.11.0
collected 13 items / 4 deselected / 9 selected                                 

tests/test_regression.py::test_no_duplicate_top_level_function_definitions PASSED [ 11%]
tests/test_regression.py::test_all_source_files_parse PASSED             [ 22%]
tests/test_regression.py::test_precedent_library_structure PASSED        [ 33%]
tests/test_regression.py::test_docking_targets_have_caveat_field PASSED  [ 44%]
tests/test_regression.py::test_agent_required_functions_exist PASSED     [ 55%]
tests/test_regression.py::test_batch_iterative_fix_loop_signature_has_debate_params PASSED [ 66%]
tests/test_regression.py::test_molecule_editor_debate_rounds_r

In [15]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add destructive-edit guard (reject candidates removing >=50% of heavy atoms) and regression tests" && git push

[main 7d63dfb] Add destructive-edit guard (reject candidates removing >=50% of heavy atoms) and regression tests
 2 files changed, 42 insertions(+)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 1.50 KiB | 1.50 MiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   068a7e6..7d63dfb  main -> main


In [17]:
!git log --oneline -3

7d63dfb (HEAD -> main, origin/main, origin/HEAD) Add destructive-edit guard (reject candidates removing >=50% of heavy atoms) and regression tests
068a7e6 Add slow integration test stubs (Qwen client-dependent, skipped outside notebook context) to test_regression.py
145835b Fix audit trail gap: debate rounds now recorded for all attempts (approved/rejected/escalated), not just the final approved one
